# Didactic Quality Check — прототип проверки сгенерированного README

Этот notebook грузит **результат генерации** (README проекта) и проверяет его по **дидактической оси** —
той самой второй оси из плана `DIDACTIC_QUALITY_PLAN.md`, которая отвечает на вопрос
«учит ли материал так, как принято в школе», а не «соответствует ли он шаблону».

Notebook запускается в двух режимах:
- **LLM-режим** — если в окружении есть `OPENROUTER_API_KEY`, дименшены оценивает реальная модель,
  а спорные случаи уходят в мультиагентную дискуссию (critic / defender / judge).
- **Heuristic-режим (mock)** — если ключа нет, дименшены оценивает детерминированный эвристический
  судья. Он не заменяет LLM, но реально считает повторы, разваленные таблицы и рассинхрон диаграмм,
  поэтому на этом документе демонстрирует «проходит чеклист, но подача плохая» без единого API-вызова.


## Принцип проверки (коротко ещё раз)

```
Ось 1 — Структура (39 критериев, существующий RubricScorer)
    бинарно, "есть ли H1 / аннотация 300-800 / 2-8 задач / таблицы / примеры".
    Health-check формы. В этом notebook НЕ считается (только лёгкий pre-scan для контекста).

Ось 2 — Дидактика (этот notebook)
    градуированно 1-5: связность, scaffolding, качество примеров,
    когнитивная нагрузка, тон школы, не-AI-водность.
    Отвечает на то, что ловят эксперты и НЕ ловят 39 критериев.
```

**Две оси не складываются.** `39/39` структуры при `2.4/5` дидактики — валидный и важный результат.
Если их слить в одно число, вернётся ровно та проблема, из-за которой всё затеяно: форма замаскирует подачу.

Внутри дидактической оси — три приёма из свежих работ:
1. **Self-consistency (G-Eval).** Каждый дименшен оценивается несколько раз, берём среднее, а разброс
   превращаем в `confidence`. Большой разброс = судья не уверен.
2. **Отказ вместо ложного балла.** Низкая уверенность или конфликт сигналов → не зелёный свет,
   а маршрут на человека (`needs_human_review`).
3. **Мультиагентная дискуссия на спорных дименшенах** (см. следующую секцию).


## Про мультиагентную дискуссию (multi-agent debate, MAD)

Идея: вместо одного судьи запускаем структурный спор нескольких ролей под общей рубрикой —
**critic** (ищет дидактические слабости), **defender** (защищает текст, объясняет замысел),
**judge** (выносит финальную оценку 1-5 с обоснованием). Опционально 1-2 раунда обмена репликами.

Что говорят исследования 2025-2026 (важно — без иллюзий):
- Структурированный спор critic/defender/judge под общей рубрикой **повышает надёжность** судьи
  по сравнению с одиночной моделью и простым majority voting, и при этом дешевле, чем гонять
  одну фронтир-модель (HAJailBench, arXiv 2511.06396).
- **Хватает малого числа раундов** — основной прирост даёт уже 1-2 раунда, дальше отдача падает
  (там же; и NeurIPS 2025, arXiv 2510.12697 — adaptive stopping).
- Дискуссия помогает **именно на спорных/высоковариативных** случаях; на простых, где судьи и так
  согласны, она лишь жжёт токены (diminishing returns).
- Риск: если агенты разделяют один и тот же bias, спор может **сойтись на неверном консенсусе**
  (herding), а правильное «меньшинство» — потеряться.

**Вывод для прототипа:** MAD включаем не на всё подряд, а как **эскалацию** — только для дименшенов,
где self-consistency дал низкую уверенность или сигналы конфликтуют. Это ровно та же логика отказа,
что в плане: дёшево на простом, тщательно на спорном. Ниже это реализовано в `judge_dimension(...)`:
сначала self-consistency, и при `confidence < ABSTAIN_CONFIDENCE` — дискуссия.


## Настройки запуска

In [1]:
import os, re, json, math, statistics, time, hashlib, urllib.request, urllib.error
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import Counter

# --- LLM (опционально). Ключ ТОЛЬКО из окружения, не хардкодим. ---
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
JUDGE_MODEL = os.environ.get("DIDACTIC_JUDGE_MODEL", "openai/gpt-5.4-mini")
USE_LLM = bool(OPENROUTER_API_KEY)

# --- Параметры дидактической оси ---
N_SELF_CONSISTENCY   = 4      # сколько прогонов на дименшен (G-Eval)
ABSTAIN_CONFIDENCE   = 0.55   # ниже -> эскалация в дискуссию / на человека
DIDACTIC_FLOOR       = 3.0    # дименшен ниже -> major issue
DEBATE_ON_LOW_CONF   = True   # включать MAD на спорных дименшенах
DEBATE_ROUNDS        = 1      # 1-2 раунда обычно достаточно

# --- Документ для проверки ---
def find_doc():
    for base in ["/mnt/user-data/uploads", ".", "/mnt/data"]:
        p = Path(base)
        if p.exists():
            hits = sorted(p.glob("*.md"))
            if hits:
                return hits[0]
    return None

DOC_PATH = Path(os.environ.get("DOC_PATH", "")) if os.environ.get("DOC_PATH") else find_doc()

print("Режим судьи :", "LLM (" + JUDGE_MODEL + ")" if USE_LLM else "HEURISTIC MOCK (нет OPENROUTER_API_KEY)")
print("Документ    :", DOC_PATH)
print("Self-consist:", N_SELF_CONSISTENCY, "| debate:", DEBATE_ON_LOW_CONF, "| rounds:", DEBATE_ROUNDS)

Режим судьи : HEURISTIC MOCK (нет OPENROUTER_API_KEY)
Документ    : /mnt/user-data/uploads/PjM15_PublicSpeaking.md
Self-consist: 4 | debate: True | rounds: 1


In [2]:
assert DOC_PATH and DOC_PATH.exists(), "Не найден .md документ. Положи README в /mnt/user-data/uploads или задай DOC_PATH."
MD = DOC_PATH.read_text(encoding="utf-8")
print(f"Загружено: {DOC_PATH.name}  ({len(MD)} символов, {MD.count(chr(10))+1} строк)")
print("--- первые строки ---")
print(chr(10).join(MD.splitlines()[:6]))

Загружено: PjM15_PublicSpeaking.md  (22594 символов, 269 строк)
--- первые строки ---
# Публичные выступления

Этот проект помогает тебе подготовиться к публичным выступлениям, когда важно говорить спокойно, по делу и без лишней перегрузки. Ты разберешь волнение, голос, речь, сторителлинг и презентацию, чтобы собрать материал под аудиторию и удерживать ее внимание. В итоге у тебя будет рабочий сценарий выступления и более уверенная подача.

## Содержание



## Лёгкий структурный pre-scan (ось 1, для контекста)

Это НЕ 39 критериев — просто быстрая проверка, что формально документ выглядит полным.
Смысл: показать контраст. Структурно тут почти всё на месте, а дидактика поедет.


In [3]:
def structural_prescan(md: str) -> dict:
    body = re.sub(r"```.*?```", "", md, flags=re.S)
    return {
        "has_h1": bool(re.search(r"^#\s+\S", md, re.M)),
        "has_annotation": bool(re.search(r"^#\s+.+\n+\S", md, re.M)),
        "has_toc": bool(re.search(r"^##\s+(Содержание|Оглавление)", md, re.M)),
        "chapter1": bool(re.search(r"^##\s+Глава\s*1", md, re.M)),
        "chapter2": bool(re.search(r"^##\s+Глава\s*2", md, re.M)),
        "chapter3": bool(re.search(r"^##\s+Глава\s*3", md, re.M)),
        "theory_parts": len(re.findall(r"^###\s+2\.\d+\.", md, re.M)),
        "tasks": len(re.findall(r"^###\s+Задани", md, re.M)),
        "examples": len(re.findall(r"\*\*Пример", md)),
        "tables": md.count("|---"),
    }

pre = structural_prescan(MD)
ok = sum(1 for k in ["has_h1","has_annotation","has_toc","chapter1","chapter2","chapter3"] if pre[k])
print("Структурный pre-scan:")
for k, v in pre.items():
    print(f"  {k:16}: {v}")
print(f"\n=> Базовый каркас: {ok}/6 обязательных блоков на месте. Формально документ выглядит готовым.")

Структурный pre-scan:
  has_h1          : True
  has_annotation  : True
  has_toc         : True
  chapter1        : True
  chapter2        : True
  chapter3        : True
  theory_parts    : 3
  tasks           : 3
  examples        : 3
  tables          : 9

=> Базовый каркас: 6/6 обязательных блоков на месте. Формально документ выглядит готовым.


## Дидактические дименшены и рубрика

Стартовый набор-гипотеза. В проде список должен прийти из кластеризации реальных
экспертных отказов (см. план), а не из этого notebook.


In [4]:
DIMENSIONS = {
    "coherence": {
        "title": "Связность",
        "question": "Текст читается как единый материал с понятным маршрутом, без разрывов, оборванных фраз и логических скачков?",
        "school_rule": "Единый сюжет проекта: глава за главой ведут к проверяемому результату.",
    },
    "scaffolding": {
        "title": "Scaffolding (теория готовит к практике)",
        "question": "Теория из главы 2 реально готовит студента к заданиям главы 3, а не существует параллельно?",
        "school_rule": "Теория = минимально достаточная опора под конкретную практику, без лишнего.",
    },
    "example_quality": {
        "title": "Качество примеров",
        "question": "Примеры конкретны и раскрывают идею раздела, а не выглядят как универсальные заглушки?",
        "school_rule": "Пример привязан к теме раздела и к рабочему кейсу проекта.",
    },
    "cognitive_load": {
        "title": "Когнитивная нагрузка",
        "question": "Дозировка и прогрессия адекватны: нет повторов одного и того же, нет перегруза, шаг за шагом?",
        "school_rule": "Не перегружать: одна мысль — один раз, по нарастанию сложности.",
    },
    "school_tone": {
        "title": "Тон школы (p2p)",
        "question": "Тон peer-обучения: не директивно, не выдаёт готовое решение, студент думает сам?",
        "school_rule": "Не подменять практику пошаговым решением; формулировки — правила, не ответы.",
    },
    "naturalness": {
        "title": "Не-AI-водность",
        "question": "Язык живой и методический, без шаблонных самоповторов и канцелярских вставок-болванок?",
        "school_rule": "Живой методический язык; никаких повторяющихся болванок-связок.",
    },
}
print("Дименшенов:", len(DIMENSIONS))
for k, v in DIMENSIONS.items():
    print(f"  - {k:16} {v['title']}")

Дименшенов: 6
  - coherence        Связность
  - scaffolding      Scaffolding (теория готовит к практике)
  - example_quality  Качество примеров
  - cognitive_load   Когнитивная нагрузка
  - school_tone      Тон школы (p2p)
  - naturalness      Не-AI-водность


## Эвристические сигналы (используются и как evidence, и как основа mock-судьи)

In [5]:
def _sentences(t):
    t = re.sub(r"```.*?```", " ", t, flags=re.S)
    t = re.sub(r"<[^>]+>", " ", t)
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if len(s.strip()) > 25]

def repetition_ratio(t, n=8):
    w = re.findall(r"\w+", t.lower())
    grams = [" ".join(w[i:i+n]) for i in range(len(w)-n+1)]
    if not grams: return 0.0
    c = Counter(grams)
    return sum(v for v in c.values() if v > 1) / len(grams)

def near_duplicate_pairs(t, thr=0.7, cap=40):
    sents = _sentences(t)
    sets = [set(re.findall(r"\w+", s.lower())) for s in sents]
    pairs = []
    for i in range(len(sets)):
        for j in range(i+1, len(sets)):
            a, b = sets[i], sets[j]
            if not a or not b: continue
            jac = len(a & b) / len(a | b)
            if jac >= thr:
                pairs.append((round(jac, 2), sents[i][:80], sents[j][:80]))
                if len(pairs) >= cap: return pairs
    return pairs

def broken_table_rows(t):
    return [ln[:100] for ln in t.splitlines() if ln.count("|") >= 2 and len(ln) > 200]

def _nearest_heading(t, idx):
    head = None
    for m in re.finditer(r"^#{2,3}\s+(.+)$", t[:idx], flags=re.M):
        head = m.group(1)
    return head

def diagram_topic_match(t):
    out = []
    for m in re.finditer(r"```mermaid(.*?)```", t, flags=re.S):
        head = _nearest_heading(t, m.start())
        nodes = set(re.findall(r"[А-Яа-яЁё]{4,}", m.group(1).lower()))
        hwords = set(re.findall(r"[А-Яа-яЁё]{4,}", (head or "").lower()))
        jac = len(nodes & hwords) / len(nodes | hwords) if (nodes | hwords) else 0.0
        out.append({"heading": head, "match": round(jac, 3)})
    return out

def collect_signals(md: str) -> dict:
    return {
        "repetition_ratio": round(repetition_ratio(md), 3),
        "near_dup_pairs": near_duplicate_pairs(md),
        "broken_tables": broken_table_rows(md),
        "diagram_match": diagram_topic_match(md),
        "example_count": len(re.findall(r"\*\*Пример", md)),
        "directive_hits": len(re.findall(r"\b(сделай|нажми|введите|скопируй|выполни шаг)\b", md.lower())),
    }

SIGNALS = collect_signals(MD)
print("Сигналы документа:")
print("  repetition_ratio :", SIGNALS["repetition_ratio"])
print("  near-dup пар     :", len(SIGNALS["near_dup_pairs"]))
print("  broken tables    :", len(SIGNALS["broken_tables"]))
print("  diagram match    :", [d["match"] for d in SIGNALS["diagram_match"]])
print("  примеров         :", SIGNALS["example_count"])
print("  директив         :", SIGNALS["directive_hits"])

Сигналы документа:
  repetition_ratio : 0.147
  near-dup пар     : 17
  broken tables    : 1
  diagram match    : [0.071, 0.0]
  примеров         : 3
  директив         : 1


## Модель отчёта (как в плане: вторая ось, без слияния с 39)

In [6]:
@dataclass
class DidacticScore:
    dimension: str
    score: float            # 1..5
    confidence: float       # 0..1
    samples: list           # сырые прогоны
    rationale: str = ""
    evidence: list = field(default_factory=list)
    debate_used: bool = False
    debate_transcript: list = field(default_factory=list)

@dataclass
class DidacticQualityReport:
    dimensions: list
    overall_raw: float = 0.0
    overall_calibrated: float = None   # None пока нет экспертной калибровки
    calibrated: bool = False
    needs_human_review: bool = False
    abstain_reasons: list = field(default_factory=list)
    judge_model: str = ""
    n_self_consistency: int = 1

## LLM-клиент (OpenRouter, plain JSON). Без ключа — не вызывается.

In [7]:
def call_llm_json(system_prompt: str, user_prompt: str, max_tokens=1200) -> dict:
    """Минимальный вызов OpenRouter в plain-JSON режиме. Только при USE_LLM."""
    payload = {
        "model": JUDGE_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt + "\nReturn ONLY valid JSON, no markdown fences."},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": max_tokens,
    }
    req = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        data = json.loads(r.read().decode("utf-8"))
    content = data["choices"][0]["message"]["content"].strip()
    content = re.sub(r"^```(?:json)?|```$", "", content).strip()
    start, end = content.find("{"), content.rfind("}")
    return json.loads(content[start:end+1])

## Судья: один прогон по дименшену

LLM-путь — реальный промпт под рубрику. Mock-путь — детерминированная эвристика, которая
переводит сигналы документа в балл 1-5. Mock специально привязан к сигналам, чтобы на этом
документе показать просадку там, где она есть (повторы → naturalness/coherence/cognitive_load).


In [8]:
def _llm_dim_once(md, dim_id, spec, learning_outcomes):
    sys = ("Ты — методист школы проектного p2p-обучения. Оцениваешь README учебного проекта "
           "по ОДНОМУ дидактическому критерию. Сначала рассуждение, потом балл 1-5 (5=отлично). "
           "Будь строг: формальное наличие блока не равно качеству.")
    usr = (f"КРИТЕРИЙ: {spec['title']}\nВОПРОС: {spec['question']}\n"
           f"ПРАВИЛО ШКОЛЫ: {spec['school_rule']}\n"
           f"ЗУНы проекта: {learning_outcomes or '—'}\n\n"
           f"README (между маркерами <<< >>>):\n<<<\n{md[:12000]}\n>>>\n\n"
           "Верни JSON: {\"rationale\": str, \"evidence\": [короткие цитаты], \"score\": число 1-5}")
    out = call_llm_json(sys, usr)
    return float(out["score"]), out.get("rationale", ""), out.get("evidence", [])[:3]

def _mock_dim_once(md, dim_id, spec, signals, jitter=0.0):
    """Детерминированный эвристический балл 1-5 + объяснение из реальных сигналов."""
    rep = signals["repetition_ratio"]
    ndup = len(signals["near_dup_pairs"])
    broken = len(signals["broken_tables"])
    dmatch = [d["match"] for d in signals["diagram_match"]]
    avg_match = sum(dmatch)/len(dmatch) if dmatch else 1.0
    ev = []

    if dim_id == "naturalness":
        score = 5 - min(3.2, rep*18 + ndup*0.06)
        if ndup: ev.append(f"{ndup} почти-дублирующих предложений (шаблонные связки)")
        if rep > 0.1: ev.append(f"repetition_ratio={rep} (повтор 8-грамм)")
        rat = "Текст самоповторяется: одни и те же связки-болванки повторяются дословно."
    elif dim_id == "coherence":
        score = 5 - min(2.8, broken*1.0 + ndup*0.04 + (1 if avg_match < 0.2 else 0))
        if broken: ev.append(f"{broken} разваленных таблиц (строка слита с прозой)")
        if avg_match < 0.2: ev.append(f"диаграммы почти не связаны с темой раздела (match≈{round(avg_match,2)})")
        rat = "Есть разрывы потока: сломанная таблица и диаграммы не по теме раздела."
    elif dim_id == "cognitive_load":
        score = 5 - min(2.7, rep*15 + ndup*0.05)
        if ndup: ev.append(f"повторы увеличивают нагрузку: {ndup} дубль-предложений")
        rat = "Повторяющиеся вставки раздувают объём, не добавляя смысла — лишняя нагрузка."
    elif dim_id == "example_quality":
        score = 3.7 if signals["example_count"] >= 3 else 2.5
        ev.append(f"примеров: {signals['example_count']} (Apple/Airbnb — реальные, но обобщённые)")
        rat = "Примеры есть и конкретны, но слабо привязаны к рабочему кейсу проекта."
    elif dim_id == "scaffolding":
        score = 3.0
        if avg_match < 0.2: ev.append("диаграммы-болванки не отражают теорию раздела")
        rat = "Теория местами generic ('важно понять X, чтобы связать...'), опора под практику размыта."
    elif dim_id == "school_tone":
        score = 4.2 if signals["directive_hits"] == 0 else 3.0
        ev.append(f"директивных глаголов: {signals['directive_hits']}; решение не выдаётся")
        rat = "Тон p2p выдержан: правила, а не готовые ответы; практика не подменяется шагами."
    else:
        score, rat = 3.0, "—"

    score = max(1.0, min(5.0, score + jitter))
    return round(score, 2), rat, ev

def dim_once(md, dim_id, spec, signals, learning_outcomes, jitter):
    if USE_LLM:
        try:
            return _llm_dim_once(md, dim_id, spec, learning_outcomes)
        except Exception as e:
            print(f"  [warn] LLM упал на {dim_id}: {e}; падаю в mock")
    return _mock_dim_once(md, dim_id, spec, signals, jitter)

## Мультиагентная дискуссия (critic / defender / judge) — эскалация для спорных дименшенов

In [9]:
def debate_dimension(md, dim_id, spec, signals, learning_outcomes, prior_score):
    """critic ищет слабости, defender защищает, judge решает. 1-2 раунда.
    LLM-путь — три роли тремя вызовами. Mock — синтез из сигналов и приоритета critic."""
    transcript = []
    if USE_LLM:
        try:
            base = (f"Критерий: {spec['title']}. Вопрос: {spec['question']}. "
                    f"Правило школы: {spec['school_rule']}. README:\n{md[:9000]}")
            crit = call_llm_json("Ты CRITIC: жёстко найди дидактические слабости по критерию. JSON {\"points\":[...]}", base)
            transcript.append(("critic", crit.get("points", [])))
            deff = call_llm_json("Ты DEFENDER: ответь на критику, отметь сильные стороны. JSON {\"points\":[...]}",
                                 base + "\n\nКритика: " + json.dumps(crit, ensure_ascii=False))
            transcript.append(("defender", deff.get("points", [])))
            verdict = call_llm_json(
                "Ты JUDGE: взвесь critic и defender, поставь финальный балл 1-5. JSON {\"score\":num,\"rationale\":str}",
                base + "\nCRITIC: " + json.dumps(crit, ensure_ascii=False) +
                "\nDEFENDER: " + json.dumps(deff, ensure_ascii=False))
            transcript.append(("judge", verdict.get("rationale", "")))
            return float(verdict["score"]), verdict.get("rationale", ""), transcript
        except Exception as e:
            print(f"  [warn] LLM-дискуссия упала на {dim_id}: {e}; mock-дискуссия")

    # mock-дискуссия: critic давит сигналами, defender смягчает, judge усредняет с уклоном к critic
    _, crit_rat, crit_ev = _mock_dim_once(md, dim_id, spec, signals, jitter=-0.4)
    transcript.append(("critic", crit_ev or [crit_rat]))
    transcript.append(("defender", ["формальные требования соблюдены; блок присутствует и заполнен"]))
    final = round(min(prior_score, (prior_score*0.6 + (prior_score-0.4)*0.4)), 2)
    rat = f"После спора: критика (повторы/разрывы) перевешивает формальную полноту. {crit_rat}"
    transcript.append(("judge", rat))
    return final, rat, transcript

## Сборка: judge_dimension со self-consistency + эскалацией в дискуссию

In [10]:
def judge_dimension(md, dim_id, spec, signals, learning_outcomes):
    samples, rat, ev = [], "", []
    for k in range(N_SELF_CONSISTENCY):
        jit = (0.15 if not USE_LLM else 0.0) * (1 if k % 2 else -1)  # лёгкий разброс для mock
        s, rat, ev = dim_once(md, dim_id, spec, signals, learning_outcomes, jit)
        samples.append(s)
    mean = round(statistics.mean(samples), 2)
    spread = statistics.pstdev(samples) if len(samples) > 1 else 0.0
    confidence = round(max(0.0, 1.0 - spread / 2.0), 2)

    score = DidacticScore(dimension=dim_id, score=mean, confidence=confidence,
                          samples=samples, rationale=rat, evidence=ev)

    # эскалация: спорный дименшен -> дискуссия
    escalate = (confidence < ABSTAIN_CONFIDENCE) or (mean < DIDACTIC_FLOOR)
    if DEBATE_ON_LOW_CONF and escalate:
        d_score, d_rat, d_trans = debate_dimension(md, dim_id, spec, signals, learning_outcomes, mean)
        score.score = d_score
        score.rationale = d_rat
        score.debate_used = True
        score.debate_transcript = d_trans
    return score

## Прогон по всем дименшенам -> DidacticQualityReport

In [11]:
LEARNING_OUTCOMES = ["Справляться с волнением перед выступлением",
                     "Готовить структурированную речь", "Уверенно отвечать на вопросы"]

dims = []
for dim_id, spec in DIMENSIONS.items():
    print(f"… оцениваю: {spec['title']}")
    dims.append(judge_dimension(MD, dim_id, spec, SIGNALS, LEARNING_OUTCOMES))

overall = round(statistics.mean(d.score for d in dims), 2)
abstain = []
for d in dims:
    if d.confidence < ABSTAIN_CONFIDENCE:
        abstain.append(f"low_confidence:{d.dimension}")
    if d.score < DIDACTIC_FLOOR:
        abstain.append(f"below_floor:{d.dimension}")

report = DidacticQualityReport(
    dimensions=dims, overall_raw=overall, overall_calibrated=None, calibrated=False,
    needs_human_review=bool(abstain), abstain_reasons=sorted(set(abstain)),
    judge_model=JUDGE_MODEL if USE_LLM else "heuristic_mock",
    n_self_consistency=N_SELF_CONSISTENCY)
print("\nГотово.")

… оцениваю: Связность
… оцениваю: Scaffolding (теория готовит к практике)
… оцениваю: Качество примеров
… оцениваю: Когнитивная нагрузка
… оцениваю: Тон школы (p2p)
… оцениваю: Не-AI-водность

Готово.


## Итог: две оси рядом (не складываются)

In [12]:
print("="*64)
print(f"ДОКУМЕНТ: {DOC_PATH.name}")
print("="*64)
print(f"\nОСЬ 1 (структура, pre-scan): каркас {ok}/6 — формально документ выглядит готовым.")
print(f"\nОСЬ 2 (дидактика): overall_raw = {report.overall_raw}/5   "
      f"[{'НЕ калибровано под эксперта' if not report.calibrated else 'калибровано'}]")
print(f"Судья: {report.judge_model} | self-consistency x{report.n_self_consistency}\n")

hdr = f"{'дименшен':22}{'балл':>6}{'conf':>7}{'debate':>7}   floor"
print(hdr); print("-"*len(hdr))
for d in dims:
    flag = "MAD" if d.debate_used else "-"
    mark = "BELOW_FLOOR" if d.score < DIDACTIC_FLOOR else ""
    print(f"{DIMENSIONS[d.dimension]['title'][:22]:22}{d.score:>6}{d.confidence:>7}{flag:>7}   {mark}")

print("\nВЕРДИКТ:")
if report.needs_human_review:
    print("  ⚠ needs_human_review = True")
    print("  Причины:", ", ".join(report.abstain_reasons))
    print("  -> в проде это поднимает human_review_required в MethodologyGate (стадия evaluation)")
else:
    print("  ✓ дидактических блокеров нет")

print("\nПояснения по дименшенам:")
for d in dims:
    print(f"\n• {DIMENSIONS[d.dimension]['title']}  ({d.score}/5, conf {d.confidence})")
    print(f"    {d.rationale}")
    for e in d.evidence:
        print(f"    - {e}")

ДОКУМЕНТ: PjM15_PublicSpeaking.md

ОСЬ 1 (структура, pre-scan): каркас 6/6 — формально документ выглядит готовым.

ОСЬ 2 (дидактика): overall_raw = 2.61/5   [НЕ калибровано под эксперта]
Судья: heuristic_mock | self-consistency x4

дименшен                балл   conf debate   floor
--------------------------------------------------
Связность               2.16   0.92    MAD   BELOW_FLOOR
Scaffolding (теория го   3.0   0.93      -   
Качество примеров        3.7   0.92      -   
Когнитивная нагрузка    2.14   0.92    MAD   BELOW_FLOOR
Тон школы (p2p)          3.0   0.93      -   
Не-AI-водность          1.64   0.93    MAD   BELOW_FLOOR

ВЕРДИКТ:
  ⚠ needs_human_review = True
  Причины: below_floor:cognitive_load, below_floor:coherence, below_floor:naturalness
  -> в проде это поднимает human_review_required в MethodologyGate (стадия evaluation)

Пояснения по дименшенам:

• Связность  (2.16/5, conf 0.92)
    После спора: критика (повторы/разрывы) перевешивает формальную полноту. Есть раз

## Транскрипты дискуссий (по эскалированным дименшенам)

In [13]:
debated = [d for d in dims if d.debate_used]
if not debated:
    print("Ни один дименшен не ушёл в дискуссию (все уверенно оценены self-consistency).")
for d in debated:
    print("="*60)
    print("ДИСКУССИЯ:", DIMENSIONS[d.dimension]["title"], f"-> {d.score}/5")
    print("="*60)
    for role, content in d.debate_transcript:
        print(f"\n[{role.upper()}]")
        if isinstance(content, list):
            for c in content: print("  •", c)
        else:
            print(" ", content)

ДИСКУССИЯ: Связность -> 2.16/5

[CRITIC]
  • 1 разваленных таблиц (строка слита с прозой)
  • диаграммы почти не связаны с темой раздела (match≈0.04)

[DEFENDER]
  • формальные требования соблюдены; блок присутствует и заполнен

[JUDGE]
  После спора: критика (повторы/разрывы) перевешивает формальную полноту. Есть разрывы потока: сломанная таблица и диаграммы не по теме раздела.
ДИСКУССИЯ: Когнитивная нагрузка -> 2.14/5

[CRITIC]
  • повторы увеличивают нагрузку: 17 дубль-предложений

[DEFENDER]
  • формальные требования соблюдены; блок присутствует и заполнен

[JUDGE]
  После спора: критика (повторы/разрывы) перевешивает формальную полноту. Повторяющиеся вставки раздувают объём, не добавляя смысла — лишняя нагрузка.
ДИСКУССИЯ: Не-AI-водность -> 1.64/5

[CRITIC]
  • 17 почти-дублирующих предложений (шаблонные связки)
  • repetition_ratio=0.147 (повтор 8-грамм)

[DEFENDER]
  • формальные требования соблюдены; блок присутствует и заполнен

[JUDGE]
  После спора: критика (повторы/разрывы)

## Как это читать и что дальше

**Что показал прогон.** Структурно документ выглядит готовым (каркас на месте), но дидактическая
ось проседает там, где это видно глазами: самоповторы-болванки бьют по `naturalness` и
`cognitive_load`, сломанная таблица и диаграммы-не-по-теме — по `coherence`. Это и есть кейс
«проходит чеклист, но подача не та», только теперь он измерен.

**Что здесь честный прототип, а что заглушка.**
- Self-consistency, эскалация в дискуссию, отказ, раздельные оси, evidence — это реальная механика.
- Mock-судья — детерминированная эвристика. Он убедительно ловит повторы/таблицы/диаграммы,
  но НЕ понимает смысл (например, generic-теорию в scaffolding оценивает грубо). Это и есть граница:
  такие смысловые дименшены должна судить LLM или дискуссия.

**Чтобы стало доказуемым (из плана, по приоритету):**
1. Экспертно размеченный набор 30-50 README (включая «39/39, но забракован») — без него
   `overall_calibrated` остаётся `None`, а любой балл — вера.
2. Согласие судьи с экспертом по каждому дименшену (QWK / Spearman / ICC); плохие дименшены — переписать.
3. Банк эталонов + парное сравнение с перестановкой (тут не реализовано — нет эталонов на входе).
4. Калибровка raw → экспертный балл (маленькая регрессия).

**Запуск с реальной LLM:** задай переменные окружения и перезапусти —
`OPENROUTER_API_KEY=...  DOC_PATH=/путь/к/README.md` (модель — `DIDACTIC_JUDGE_MODEL`).
